In [3]:
import os
from typing import List, Dict, Any
import pandas as pd
from langchain_core.documents import Document # to utlizie and understand the interface DOCUMENT type

# Document Structure in LangChain

A Document stores the main text content along with useful metadata.
It helps organize information from different sources in a consistent format.
This makes it easier to retrieve, filter, and answer questions accurately.

Why it is used: In RAG pipelines, Documents are the standard unit passed into chunking, embeddings, and vector search. They keep context such as source, page, and author, making retrieval more traceable and reliable.

In [4]:
doc = Document(
    page_content= "Hi, this is my first understanding of most important concept of langchain integrations, it is DOCUMENT structure",
    metadata={
            "source":"example.txt",
            "page":1,
            "author":"It's me",
            "date_created":"2026-10-05",
            "cutom_field":"Beginnig today"   
        }
)

print("Document Structure")

print(f'Type of doc:',type(doc))

print(f"Content :{doc.page_content}")
print(f"Metadata :{doc.metadata}")

# Why metadata matters:
print("\n📝 Metadata is crucial for:")
print("- Filtering search results")
print("- Tracking document sources")
print("- Providing context in responses")
print("- Debugging and auditing")

Document Structure
Type of doc: <class 'langchain_core.documents.base.Document'>
Content :Hi, this is my first understanding of most important concept of langchain integrations, it is DOCUMENT structure
Metadata :{'source': 'example.txt', 'page': 1, 'author': "It's me", 'date_created': '2026-10-05', 'cutom_field': 'Beginnig today'}

📝 Metadata is crucial for:
- Filtering search results
- Tracking document sources
- Providing context in responses
- Debugging and auditing


# Ingesting and Parsing Text data using Document Loaders

## TEXT DATA Ingestion

In [5]:
import os 
# making a textfile to test the loading process using textloader
os.makedirs("data/text_files",exist_ok=True)

In [6]:
sample_text = {
"data/text_files/langchain_intro.txt" : """

LANGCHAIN INTRODUCTION SUMMARY

1. OVERVIEW
LangChain is an open-source framework for building applications with 
Large Language Models (LLMs) like GPT-4 or Gemini. It provides standard 
interfaces to connect models with data, memory, and external tools.

2. WHY USE LANGCHAIN?
- Unified API: Switch between different LLM providers easily.
- Modularity: Swap out prompts, models, and vector stores seamlessly.
- Reduced Code: Turn 100+ lines of raw API calls into short, readable chains.

3. CORE MODULES
- Model I/O: Prompt templates, language models, and output parsers.
- Retrieval: Document loaders, text splitters, embeddings, and vector stores.
- Chains: Combining multiple components for sequential tasks (LCEL).
- Agents: Letting LLMs decide actions and use tools dynamically.

4. COMMON USE CASES
- Retrieval-Augmented Generation (RAG) for private document Q&A.
- Conversational chatbots with memory of past chats.
- Autonomous AI agents that execute multi-step tasks.
""",
"data/text_files/rag_intro.txt":
"""Retrieval-Augmented Generation (RAG) is an artificial intelligence technique that improves large language model (LLM) accuracy by pulling facts from external data sources before writing an answer.
How RAG Works
• User Query: A user types a question into the system.
• Information Retrieval: The system searches an external database or document collection for text that matches the question.
• Prompt Augmentation: The system combines the user's original question with the retrieved facts into a single instruction.
• Response Generation: The language model reads the added facts and writes a clear, factual reply.
Key Benefits
• Fewer Mistakes: It stops the AI from guessing or making up false facts (hallucinations) by grounding answers in real documents.
• Fresh Data: It bypasses the fixed end date of a model's training data by reading live or updated files.
• Lower Cost: It avoids the expensive process of retraining or fine-tuning an entire AI model.
"""
}
for path,txt_content in sample_text.items():
    with open(path,'w',encoding = "utf-8") as f:
        f.write(txt_content)

print("Sample text added successfully !!")


Sample text added successfully !!


## TextLoaders to read single file

In [7]:
# from langchain.document_loaders import TextLoader #depriciated
from langchain_community.document_loaders import TextLoader

## Loading a single text file
loader=TextLoader("data/text_files/langchain_intro.txt", encoding="utf-8") # using the same encoding to load

document = loader.load() # this will return a document type list metadata and page_content
print(f'Type of Content in the document list:',type(document[0]))
print(f"📄 Loaded {len(document)} document")
print(f"Content preview: {document[0].page_content[:100]}...")
print(f"Metadata: {document[0].metadata}")

Type of Content in the document list: <class 'langchain_core.documents.base.Document'>
📄 Loaded 1 document
Content preview: 

LANGCHAIN INTRODUCTION SUMMARY

1. OVERVIEW
LangChain is an open-source framework for building app...
Metadata: {'source': 'data/text_files/langchain_intro.txt'}


C:\Users\Dell\AppData\Local\Temp\ipykernel_9700\3878142029.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


## DirectoryLoader for loading multiple textcontent

In [8]:
from langchain_community.document_loaders import DirectoryLoader # directory loaders are used to load multiple text data from multiple file

multi_loader = DirectoryLoader(
    "data/text_files",
    glob="**/*.txt", ## Pattern to match files or paths
    loader_cls = TextLoader, ## to mention which loader class to use
    loader_kwargs= {'encoding':'utf-8'},
    show_progress=True # progress bar 
)

documents = multi_loader.load()

print(f"📁 Loaded {len(documents)} documents")
for i, doc in enumerate(documents):
    print(f"\nDocument {i+1}:")
    print(f"  Source: {doc.metadata['source']}")
    print(f"  Length: {len(doc.page_content)} characters")
    print('=='*20)

print("\n📊 DirectoryLoader Characteristics:")

print("\n✅ Advantages:")
print("  - Loads multiple files at once")
print("  - Supports glob patterns")
print("  - Progress tracking")
print("  - Recursive directory scanning")
print('=='*20)
print("\n❌ Disadvantages:")
print("  - All files must be same type")
print("  - Limited error handling per file")
print("  - Can be memory intensive for large directories")

100%|██████████| 2/2 [00:00<00:00, 148.23it/s]

📁 Loaded 2 documents

Document 1:
  Source: data\text_files\langchain_intro.txt
  Length: 980 characters

Document 2:
  Source: data\text_files\rag_intro.txt
  Length: 958 characters

📊 DirectoryLoader Characteristics:

✅ Advantages:
  - Loads multiple files at once
  - Supports glob patterns
  - Progress tracking
  - Recursive directory scanning

❌ Disadvantages:
  - All files must be same type
  - Limited error handling per file
  - Can be memory intensive for large directories


# Text Pasing techniques/Strategies
- Majorly we have 3 Text Parsing techniques effective for breaking the extracted text to chunks.
1. RecursiveCharacterTextSplitter
2. CharacterTextSplitter
3. TokenTextSplitter

In [1]:
from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
    TokenTextSplitter
)

In [13]:
print(documents)

text = documents[0].page_content

[Document(metadata={'source': 'data\\text_files\\langchain_intro.txt'}, page_content='\n\nLANGCHAIN INTRODUCTION SUMMARY\n\n1. OVERVIEW\nLangChain is an open-source framework for building applications with \nLarge Language Models (LLMs) like GPT-4 or Gemini. It provides standard \ninterfaces to connect models with data, memory, and external tools.\n\n2. WHY USE LANGCHAIN?\n- Unified API: Switch between different LLM providers easily.\n- Modularity: Swap out prompts, models, and vector stores seamlessly.\n- Reduced Code: Turn 100+ lines of raw API calls into short, readable chains.\n\n3. CORE MODULES\n- Model I/O: Prompt templates, language models, and output parsers.\n- Retrieval: Document loaders, text splitters, embeddings, and vector stores.\n- Chains: Combining multiple components for sequential tasks (LCEL).\n- Agents: Letting LLMs decide actions and use tools dynamically.\n\n4. COMMON USE CASES\n- Retrieval-Augmented Generation (RAG) for private document Q&A.\n- Conversational ch

# 1. RecursiveCharacterTextSplitter

- The recommended default text splitter that recursively works down a list of separators (typically paragraphs \n\n, lines \n, spaces , and characters "") to keep semantically related text together.
### *Pros:*
- Intelligently maintains natural language boundaries by trying to keep paragraphs, sentences, and words intact.
- Highly robust and adaptable across unstructured text, code files, PDFs, and web pages without requiring custom separator tuning.
### *Cons:*
- Computationally heavier than flat character splitters due to recursive evaluation of separators.
- Operates on raw character counts by default rather than model tokens, which can occasionally misalign with strict LLM context limits.

In [ ]:
recur_text = RecursiveCharacterTextSplitter(
    separators=["\n\n","\n"," ",""],
    chunk_size = 200,
    chunk_overlap =20,
    length_function = len,
    add_start_index = True
)
chunks_txt = recur_text.split_text(text)
chunks_doc = recur_text.split_documents(documents)

# print(chunks_txt)
print(chunks_doc[0]) 
print(f'Number of Chunks:',len(chunks_txt))
print(f'First Chunk Character:',chunks_txt[0],'....')
print(f'Understandin start index',chunks_doc[0].metadata.get('start_index'))

['LANGCHAIN INTRODUCTION SUMMARY', '1. OVERVIEW\nLangChain is an open-source framework for building applications with \nLarge Language Models (LLMs) like GPT-4 or Gemini. It provides standard', 'interfaces to connect models with data, memory, and external tools.', '2. WHY USE LANGCHAIN?\n- Unified API: Switch between different LLM providers easily.\n- Modularity: Swap out prompts, models, and vector stores seamlessly.', '- Reduced Code: Turn 100+ lines of raw API calls into short, readable chains.', '3. CORE MODULES\n- Model I/O: Prompt templates, language models, and output parsers.\n- Retrieval: Document loaders, text splitters, embeddings, and vector stores.', '- Chains: Combining multiple components for sequential tasks (LCEL).\n- Agents: Letting LLMs decide actions and use tools dynamically.', '4. COMMON USE CASES\n- Retrieval-Augmented Generation (RAG) for private document Q&A.\n- Conversational chatbots with memory of past chats.\n- Autonomous AI agents that execute multi-step t

In [47]:
print(chunks_txt[0])
print("-------------")
print(chunks_txt[1])
print("-------------")
print(chunks_txt[2])


# No overlapping is found because its the beauty of recursive text character splitter, as it has seperators :)

LANGCHAIN INTRODUCTION SUMMARY
-------------
1. OVERVIEW
LangChain is an open-source framework for building applications with 
Large Language Models (LLMs) like GPT-4 or Gemini. It provides standard
-------------
interfaces to connect models with data, memory, and external tools.


# 2.CharacterTextSplitter

- A basic splitter that splits text strictly based on a single user-defined separator character (defaulting to double newlines \n\n).

### *Pros:*
- Simple, lightweight, and predictable behavior when working with uniform documents that follow strict structural formatting.
- Highly efficient and fast computation due to the lack of complex cascading logic.
### *Cons:*
- Inflexible; if the chosen separator is missing or infrequent, it produces poorly sized chunks or fails to split effectively.
- Does not attempt to preserve semantic boundaries like sentences or words if a chunk exceeds the target size.

In [33]:
char_splitter = CharacterTextSplitter(
    separator="\n",  # Split on newlines
    chunk_size=200,  # Max chunk size in characters
    chunk_overlap=20,  # Overlap between chunks
    length_function=len  # How to measure chunk size
)

char_chunks=char_splitter.split_text(text)
print(f"Created {len(char_chunks)} chunks")
print(f"First chunk: {char_chunks[0][:100]}...")

Created 6 chunks
First chunk: LANGCHAIN INTRODUCTION SUMMARY
1. OVERVIEW
LangChain is an open-source framework for building applic...


In [32]:
print(char_chunks[0])
print("------------------")
print(char_chunks[1])
print("------------------")
print(char_chunks[2])

LANGCHAIN INTRODUCTION SUMMARY

1. OVERVIEW
LangChain is an open-source framework for building applications with 
Large Language Models (LLMs) like GPT-4 or Gemini. It provides standard 
interfaces
------------------
tandard 
interfaces to connect models with data, memory, and external tools.

2. WHY USE LANGCHAIN?
- Unified API: Switch between different LLM providers easily.
- Modularity: Swap out prompts, models
------------------
out prompts, models, and vector stores seamlessly.
- Reduced Code: Turn 100+ lines of raw API calls into short, readable chains.

3. CORE MODULES
- Model I/O: Prompt templates, language models, and o


# 3. TokenTextSplitter

- The abstract base class and interface defining how documents are split into chunks.

### *Pros:*
- Provides a standardized split_documents and split_text interface across all custom and built-in splitters.
- Includes built-in utilities for length measurement, chunk merging, and metadata preservation (such as add_start_index).
### *Cons:*
- Abstract class that cannot be instantiated directly; requires a concrete subclass implementation.
- Lacks built-in splitting logic or separator prioritization out of the box.

In [49]:
# Token Based Splitter
token_split = TokenTextSplitter(
    chunk_size = 50,
    chunk_overlap = 10,
)
tkn_chunks = token_split.split_text(text)

print(f'NUmber of chunks:',len(tkn_chunks))
print("First Chunk:"+tkn_chunks[0][:100])

NUmber of chunks: 7
First Chunk:

LANGCHAIN INTRODUCTION SUMMARY

1. OVERVIEW
LangChain is an open-source framework for building app


In [52]:
print(tkn_chunks[0])
print("------------------")
print(tkn_chunks[1])
print("------------------")
print(tkn_chunks[2])



LANGCHAIN INTRODUCTION SUMMARY

1. OVERVIEW
LangChain is an open-source framework for building applications with 
Large Language Models (LLMs) like GPT-4 or Gemini.
------------------
Ms) like GPT-4 or Gemini. It provides standard 
interfaces to connect models with data, memory, and external tools.

2. WHY USE LANGCHAIN?
- Unified API: Switch between different LLM
------------------

- Unified API: Switch between different LLM providers easily.
- Modularity: Swap out prompts, models, and vector stores seamlessly.
- Reduced Code: Turn 100+ lines of raw API calls into short, readable chains.




# 4. RecursiveCharacterTextSplitter using tiktoken
- A specialized configuration of the recursive character splitter that measures chunk lengths using tokenizer token counts (via from_tiktoken_encoder) instead of raw character lengths.
- Using tick_token to measure and splits characters based on tokens

### *Pros:*
- Guarantees that every generated chunk strictly respects model token limits (preventing embedding or LLM token overflow errors).
- Retains all semantic grouping benefits of the recursive splitting strategy while ensuring accurate sizing for specific tokenizers.
### *Cons:*
- Requires external tokenizer dependencies (tiktoken) and specific model encodings.
- Slower execution overhead due to token encoding and decoding operations during length evaluation.

### Uses
- Context Window Safety: If your prompt limit is 4,096 tokens, character-based splitting can easily exceed this limit if the text contains code, emojis, or foreign languages with high character-to-token ratios.

- Cost Efficiency: AI providers (like OpenAI) charge per token. Measuring chunks by tokens gives you predictable control over API usage costs.

- Context Preservation: Overlapping by exact token counts ensures that semantic context carries over seamlessly from one vector database retrieval chunk to the next.

In [61]:
# Use the from_tiktoken_encoder factory method to initialize the RecursiveCharacterTextSplitter with token-based chunk sizing.
# This ensures your chunks are measured by model tokens rather than raw characters:

from langchain_text_splitters import RecursiveCharacterTextSplitter

# Initialize the text splitter using tiktoken encoding
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=100,
    chunk_overlap=50,
)

# Split your text into token-bounded chunks
doc_splits = text_splitter.split_text(text)
print(len(doc_splits))
print(doc_splits[0])
print(len(doc_splits[0]))

4
LANGCHAIN INTRODUCTION SUMMARY

1. OVERVIEW
LangChain is an open-source framework for building applications with 
Large Language Models (LLMs) like GPT-4 or Gemini. It provides standard 
interfaces to connect models with data, memory, and external tools.
254


In [74]:
ai_txt = """
Artificial intelligence (AI) is intelligence demonstrated by machines, as opposed to the natural intelligence displayed by animals including humans. 
AI research has been defined as the field of study of intelligent agents, which refers to any system that perceives its environment and takes actions that maximize its chance of achieving its goals.
The term 'artificial intelligence' had previously been used to describe machines that mimic and display 'human' cognitive skills that people associate with the human mind, such as 'learning' and 'problem solving'. 
This definition has since been rejected by major AI researchers who now describe AI in terms of rationality and acting rationally, which does not limit how intelligence can be articulated.
"""

# Custom list of separators prioritized in order
custom_separators = [
    "\n\n",  # First, try to split by double line breaks (paragraphs)
    "\n",    # Next, try single line breaks
    ". ",    # Next, try end of sentences
    " ",     # Next, try spaces (words)
    ""       # Fallback: split by individual characters
]

# Initialize the splitter with tiktoken encoding
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base",  # Default encoding used by GPT-3.5 and GPT-4
    chunk_size=40,                # Max 40 tokens per chunk
    chunk_overlap=10,             # 10 tokens overlap between chunks
)

# text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
#     encoding_name="cl100k_base",
#     chunk_size=40,
#     chunk_overlap=10,
#     separators=custom_separators,
#     is_separator_regex=False 
#     )

doc_splits = text_splitter.split_text(ai_txt)

print(f"Total Chunks Created: {len(doc_splits)}\n")

for idx, chunk in enumerate(doc_splits):
    print(f"--- Chunk {idx + 1} ---")
    print(chunk.strip())
    print("-" * 20)

Total Chunks Created: 5

--- Chunk 1 ---
Artificial intelligence (AI) is intelligence demonstrated by machines, as opposed to the natural intelligence displayed by animals including humans.
--------------------
--- Chunk 2 ---
AI research has been defined as the field of study of intelligent agents, which refers to any system that perceives its environment and takes actions that maximize its chance of achieving its goals.
--------------------
--- Chunk 3 ---
The term 'artificial intelligence' had previously been used to describe machines that mimic and display 'human' cognitive skills that people associate with the human mind, such as 'learning' and 'problem
--------------------
--- Chunk 4 ---
mind, such as 'learning' and 'problem solving'.
--------------------
--- Chunk 5 ---
This definition has since been rejected by major AI researchers who now describe AI in terms of rationality and acting rationally, which does not limit how intelligence can be articulated.
--------------------


In [54]:
# 📊 Comparison
print("\n📊 Text Splitting Methods Comparison:")
print("\nCharacterTextSplitter:")
print("  ✅ Simple and predictable")
print("  ✅ Good for structured text")
print("  ❌ May break mid-sentence")
print("  Use when: Text has clear delimiters")

print("\nRecursiveCharacterTextSplitter:")
print("  ✅ Respects text structure")
print("  ✅ Tries multiple separators")
print("  ✅ Best general-purpose splitter")
print("  ❌ Slightly more complex")
print("  Use when: Default choice for most texts")

print("\nTokenTextSplitter:")
print("  ✅ Respects model token limits")
print("  ✅ More accurate for embeddings")
print("  ❌ Slower than character-based")
print("  Use when: Working with token-limited models")


📊 Text Splitting Methods Comparison:

CharacterTextSplitter:
  ✅ Simple and predictable
  ✅ Good for structured text
  ❌ May break mid-sentence
  Use when: Text has clear delimiters

RecursiveCharacterTextSplitter:
  ✅ Respects text structure
  ✅ Tries multiple separators
  ✅ Best general-purpose splitter
  ❌ Slightly more complex
  Use when: Default choice for most texts

TokenTextSplitter:
  ✅ Respects model token limits
  ✅ More accurate for embeddings
  ❌ Slower than character-based
  Use when: Working with token-limited models
